In [ ]:
from flask import Flask, render_template, redirect, url_for, request, flash
from flask_sqlalchemy import SQLAlchemy 
from flask_login import LoginManager, UserMixin, login_user, login_required, logout_user, current_user

app = Flask(__name__)

app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///database.db'
app.config['SECRET_KEY'] = 'thisissecret'

db = SQLAlchemy(app)
login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_message = "You need to login again."
login_manager.refresh_view = 'login'
login_manager.login_view = 'login'
    
class User(UserMixin, db.Model):
    __tablename__ = 'users'
    id = db.Column(db.Integer, primary_key=True, autoincrement=True)
    username = db.Column(db.String(50))
    email = db.Column(db.String(255))
    password = db.Column(db.String(80))
    
    def __repr__(self):
        return '<User %r>' % self.id

@login_manager.user_loader
def load_user(user_id):
    return User.query.get(int(user_id))

@app.route("/")
def home():
    status = "home"
    return render_template('index.html', status=status)


@app.route('/login', methods=["GET", "POST"])
def login():
    message=''
    status = "login"
    if request.method == "POST":
        username = request.form['username']
        password = request.form['password']
        print(username,password)

        user = User.query.filter_by(username=username).first()
        if not user:
            print("Wrong username or username not found.")
            flash("Wrong username or username not found.")
            return redirect(url_for('login'))
        else:
            if password == user.password:
                print("Successfully logged in.")
                login_user(user)
                flash("Successfully logged in.")
                return redirect(url_for('profile'))
            else:
                print("Wrong password.")
                flash("Wrong password.")
                return redirect(url_for('login'))           
    else:
        return render_template('login.html',status=status)

@app.route("/profile")
@login_required
def profile():
    status = "profile"
    return render_template('profile.html',username=current_user.username, status=status)

@app.route("/signup", methods=["GET", "POST"])
def signup():
    status = "signup"
    if request.method == "POST":
        username = request.form['username']
        email = request.form['email']
        password = request.form['password']
        print(username,email,password)
        new_user = User(username=username,email=email,password=password)     
        try:
            db.session.add(new_user)
            db.session.commit()
            print("Registered!")
            return redirect(url_for('home'))
        except Exception as e:
            print("Exception occurred.",e)
            return 'Fail to add user.'
    else:
        return render_template('signup.html',status=status)

@app.route("/logout")
@login_required
def logout():
    print('logout user.')
    flash('logged out.', current_user.username)
    logout_user()
    return redirect(url_for('home')) 


@app.errorhandler(404)
def page_not_found(e):
    return render_template('404.html'), 404

if __name__ == "__main__":
    app.run('localhost', 9033)

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


/Users/suen/anaconda3/lib/python3.7/site-packages/flask_sqlalchemy/__init__.py:834: FSADeprecationWarning: SQLALCHEMY_TRACK_MODIFICATIONS adds significant overhead and will be disabled by default in the future.  Set it to True or False to suppress this warning.
  'SQLALCHEMY_TRACK_MODIFICATIONS adds significant overhead and '
 * Running on http://localhost:9033/ (Press CTRL+C to quit)
127.0.0.1 - - [25/Oct/2021 13:58:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [25/Oct/2021 13:58:38] "GET /profile HTTP/1.1" 302 -
127.0.0.1 - - [25/Oct/2021 13:58:38] "GET /login?next=%2Fprofile HTTP/1.1" 200 -
127.0.0.1 - - [25/Oct/2021 13:58:39] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [25/Oct/2021 13:59:42] "GET /login HTTP/1.1" 200 -
127.0.0.1 - - [25/Oct/2021 13:59:50] "GET /profile HTTP/1.1" 302 -
127.0.0.1 - - [25/Oct/2021 13:59:50] "GET /login?next=%2Fprofile HTTP/1.1" 200 -
127.0.0.1 - - [25/Oct/2021 13:59:52] "GET /login HTTP/1.1" 200 -
127.0.0.1 - - [25/Oct/2021 14:20:57] "GET / HTTP/1.1" 200 -
